In [1]:
pip install spacy stanza spacy-stanza

In [21]:
import os
os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "0"
import sys
import numpy

# Creamos la ruta clásica que busca el archivo de Stanford
if not hasattr(numpy, 'core'):
    import types
    core_module = types.ModuleType('numpy.core')
    multiarray_module = types.ModuleType('numpy.core.multiarray')

    multiarray_module._reconstruct = numpy._core.multiarray._reconstruct
    core_module.multiarray = multiarray_module

    sys.modules['numpy.core'] = core_module
    sys.modules['numpy.core.multiarray'] = multiarray_module
    numpy.core = core_module

import torch
torch.serialization.add_safe_globals([numpy.core.multiarray._reconstruct])

# torch.load para forzar weights_only=False
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

# improtamos el resto de Libs
import spacy
import stanza
import spacy_stanza
from spacy_stanza.tokenizer import StanzaTokenizer

texto_ejemplo = "Kaixo mundua, spaCy euskaraz probatzen ari naiz."

Probamos con Spacy

In [23]:
nlp_spacy = spacy.blank("eu")
doc_spacy = nlp_spacy(texto_ejemplo)

print("\n--- Detección de Stopwords y Tokens ---")
for token in doc_spacy:
    # Comprobamos si la palabra está catalogada como stopword en euskera
    estado_stop = "Es Stopword" if token.is_stop else "No es..."
    print(f"Token: {token.text:<12} | {estado_stop}")

print("\n--- Intento de obtener POS Tagging  ---")
for token in doc_spacy:
    # Verás que devuelve una cadena vacía o valores por defecto del tokenizador
    print(f"Token: {token.text:<12} | POS detectado: '{token.pos_}'")


print("\n" + "="*60 + "\n")



--- Detección de Stopwords y Tokens ---
Token: Kaixo        | No es...
Token: mundua       | No es...
Token: ,            | No es...
Token: spaCy        | No es...
Token: euskaraz     | No es...
Token: probatzen    | No es...
Token: ari          | No es...
Token: naiz         | No es...
Token: .            | No es...

--- Intento de obtener POS Tagging  ---
Token: Kaixo        | POS detectado: ''
Token: mundua       | POS detectado: ''
Token: ,            | POS detectado: ''
Token: spaCy        | POS detectado: ''
Token: euskaraz     | POS detectado: ''
Token: probatzen    | POS detectado: ''
Token: ari          | POS detectado: ''
Token: naiz         | POS detectado: ''
Token: .            | POS detectado: ''




PRUEBA INTEGRANDO SPACY + STANZA

In [22]:
# Descargamos el modelo oficial de euskera de Stanford
print("Descargando/Verificando modelo de Euskera de Stanford...")
stanza.download("eu")

# Cargamos el pipeline que conecta Stanza dentro de spaCy
nlp_completo = spacy_stanza.load_pipeline("eu")
doc_stanza = nlp_completo(texto_ejemplo)

print("\n--- Análisis de POS Tagging con Stanza ---")
for token in doc_stanza:
    # Ahora el objeto de spaCy ya tiene los datos morfológicos inyectados por Stanza
    print(f"Token: {token.text:<12} | POS real: {token.pos_:<6} | Explicación: {token.tag_}")

Descargando/Verificando modelo de Euskera de Stanford...


INFO:stanza:Downloading default packages for language: eu (Basque) ...
INFO:stanza:File exists: /root/stanza_resources/eu/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources.
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Loading these models for language: eu (Basque):
| Processor | Package      |
----------------------------
| tokenize  | bdt          |
| pos       | bdt_nocharlm |
| lemma     | bdt_nocharlm |
| depparse  | bdt_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!



--- Análisis de POS Tagging con Stanza (¡Ahora SÍ funciona!) ---
Token: Kaixo        | POS real: INTJ   | Explicación: INTJ
Token: mundua       | POS real: NOUN   | Explicación: NOUN
Token: ,            | POS real: PUNCT  | Explicación: PUNCT
Token: spaCy        | POS real: NOUN   | Explicación: NOUN
Token: euskaraz     | POS real: NOUN   | Explicación: NOUN
Token: probatzen    | POS real: VERB   | Explicación: VERB
Token: ari          | POS real: AUX    | Explicación: AUX
Token: naiz         | POS real: VERB   | Explicación: VERB
Token: .            | POS real: PUNCT  | Explicación: PUNCT
